In [ ]:
!pip install https://github.com/opendatalab/magic-html/releases/download/magic_html-0.1.5-released/magic_html-0.1.5-py3-none-any.whl
!pip install requests

In [ ]:
#simple version

from magic_html import GeneralExtractor
import requests

# 初始化提取器
extractor = GeneralExtractor()


url = 'https://juejin.cn/post/7304867278566899764?utm_source=gold_browser_extension'
resp = requests.get(url)
html=resp.text

# 文章类型HTML提取数据
data = extractor.extract(html, base_url='https://juejin.cn')

print(data)


In [ ]:
#wechat version

from magic_html import GeneralExtractor
import requests

# Initialize the extractor
extractor = GeneralExtractor()

# URL to fetch the WeChat article
url = "https://mp.weixin.qq.com/s?__biz=MzUxODA2MjM0Ng%3D%3D&mid=2247625011&idx=1&sn=042694d7e7aeab0919917dd480358125&chksm=f8e0a3c32938d16d92d4ed94187f6435f2269e902a82321532e38be50fcaa126ab1833c8becc&scene=27"
try:
    # Fetch the HTML content of the page
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
    resp.encoding = resp.apparent_encoding  # Handle encoding
    html = resp.text
    print("HTML fetched successfully.")
except requests.RequestException as e:
    print(f"Error fetching the URL: {e}")
    html = ""

# Extract data based on WeChat article type
if html:
    try:
        data = extractor.extract(html, base_url="https://mp.weixin.qq.com", html_type="weixin")
        print("Extracted Data:", data)
    except Exception as e:
        print(f"Error during extraction: {e}")

HTML fetched successfully.
Extracted Data: {'xp_num': 'weixin', 'drop_list': False, 'html': '<div id="img-content" class="rich_media_wrp">\n          \n          <h1 class="rich_media_title " id="activity-name">“城市大脑”赋能国防动员——多地乘智慧城市发展东风推动智慧动员建设</h1>\n          <div class="rich_media_content js_underline_content\n                       autoTypeSetting24psection\n            " id="js_content" style="visibility: hidden; opacity: 0; "><p style="letter-spacing: 0.578px;white-space: normal;"><img class="rich_pages wxw-img" data-imgfileid="100141346" data-ratio="0.2064814814814815" data-src="https://mmbiz.qpic.cn/sz_mmbiz_png/ibpmnoMXsCAzrMfZua4ncu6BzyWf52r2Z13I38N1IdwNkuepOu70ib8vlXr26RESGWPCckBibQ14icbD3XbuydmLdg/640?wx_fmt=other&amp;from=appmsg&amp;wxfrom=5&amp;wx_lazy=1&amp;wx_co=1&amp;tp=webp" data-w="1080" src="https://mmbiz.qpic.cn/sz_mmbiz_png/ibpmnoMXsCAzrMfZua4ncu6BzyWf52r2Z13I38N1IdwNkuepOu70ib8vlXr26RESGWPCckBibQ14icbD3XbuydmLdg/640?wx_fmt=other&amp;from=appmsg&amp;wxfrom=5&amp;wx

In [5]:
print(data["html"])  # Outputs the main content in HTML format.

<div id="img-content" class="rich_media_wrp">
          
          <h1 class="rich_media_title " id="activity-name">“城市大脑”赋能国防动员——多地乘智慧城市发展东风推动智慧动员建设</h1>
          <div class="rich_media_content js_underline_content
                       autoTypeSetting24psection
            " id="js_content" style="visibility: hidden; opacity: 0; "><p style="letter-spacing: 0.578px;white-space: normal;"><img class="rich_pages wxw-img" data-imgfileid="100141346" data-ratio="0.2064814814814815" data-src="https://mmbiz.qpic.cn/sz_mmbiz_png/ibpmnoMXsCAzrMfZua4ncu6BzyWf52r2Z13I38N1IdwNkuepOu70ib8vlXr26RESGWPCckBibQ14icbD3XbuydmLdg/640?wx_fmt=other&amp;from=appmsg&amp;wxfrom=5&amp;wx_lazy=1&amp;wx_co=1&amp;tp=webp" data-w="1080" src="https://mmbiz.qpic.cn/sz_mmbiz_png/ibpmnoMXsCAzrMfZua4ncu6BzyWf52r2Z13I38N1IdwNkuepOu70ib8vlXr26RESGWPCckBibQ14icbD3XbuydmLdg/640?wx_fmt=other&amp;from=appmsg&amp;wxfrom=5&amp;wx_lazy=1&amp;wx_co=1&amp;tp=webp"></p><p style="text-align: justify;"><img class="rich_pages wxw-im

In [ ]:
!pip install html2text

In [ ]:
import html2text

def convert_html_to_markdown(html_content):
    """
    Convert HTML string to Markdown
    
    Args:
        html_content: String containing HTML
        
    Returns:
        String containing Markdown
    """
    # Initialize html2text converter with customized settings
    h = html2text.HTML2Text()
    h.ignore_links = False
    h.ignore_images = False
    h.ignore_tables = False
    h.body_width = 0  # Don't wrap text
    h.protect_links = True
    h.mark_code = True
    h.unicode_snob = True
    
    # Convert HTML to Markdown
    markdown_content = h.handle(html_content)
    
    return markdown_content

# Example usage
# Assuming you have the HTML in a variable called 'html'
markdown = convert_html_to_markdown(html)

# Print the markdown
print(markdown)

# Or save it to a file
with open('article.md', 'w', encoding='utf-8') as f:
    f.write(markdown)


Below is best working copy...includes images

In [ ]:
import html2text
import requests
import os
import re
import uuid
from urllib.parse import urljoin, urlparse

def convert_html_to_markdown_with_images(html_content, base_url='', output_dir='images'):
    """
    Convert HTML to Markdown and download images
    
    Args:
        html_content: String containing HTML
        base_url: Base URL for resolving relative image paths
        output_dir: Directory to save downloaded images
        
    Returns:
        String containing Markdown with local image references
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Initialize html2text converter
    h = html2text.HTML2Text()
    h.ignore_links = False
    h.ignore_images = False
    h.ignore_tables = False
    h.body_width = 0  # Don't wrap text
    h.protect_links = True
    h.mark_code = True
    h.unicode_snob = True
    
    # First, convert HTML to Markdown
    markdown_content = h.handle(html_content)
    
    # Find all image URLs in the Markdown
    # Regex pattern for Markdown image syntax: ![alt text](image_url)
    image_pattern = r'!\[.*?\]\((.*?)\)'
    image_urls = re.findall(image_pattern, markdown_content)
    
    # Download each image and replace URLs
    for img_url in image_urls:
        try:
            # Handle relative URLs
            if not img_url.startswith(('http://', 'https://')):
                img_url = urljoin(base_url, img_url)
            
            # Get file extension from URL or default to .jpg
            file_ext = os.path.splitext(urlparse(img_url).path)[1]
            if not file_ext:
                file_ext = '.jpg'
            
            # Generate a unique filename
            filename = f"{uuid.uuid4().hex}{file_ext}"
            filepath = os.path.join(output_dir, filename)
            
            # Download the image
            print(f"Downloading image: {img_url}")
            response = requests.get(img_url, stream=True)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            # Replace the URL in the Markdown content
            markdown_content = markdown_content.replace(
                f']({img_url})', 
                f']({os.path.join(output_dir, filename)})'
            )
            
            print(f"Saved image to {filepath}")
            
        except Exception as e:
            print(f"Error downloading image {img_url}: {e}")
    
    return markdown_content

# Example usage
# Assuming you have the HTML in a variable called 'html'
markdown = convert_html_to_markdown_with_images(
    html, 
    base_url='https://mp.weixin.qq.com/'  # Provide the base URL for resolving relative paths
)

# Save the Markdown to a file
with open('article_with_images.md', 'w', encoding='utf-8') as f:
    f.write(markdown)


**both extract & convert & save below**

In [ ]:
!pip install markdownify beautifulsoup4 requests matplotlib

In [14]:
from magic_html import GeneralExtractor
import requests
from markdownify import markdownify as md
import os
import re
import uuid
from urllib.parse import urljoin, urlparse
import json
from bs4 import BeautifulSoup
import time
from IPython.display import Markdown, display


In [16]:
def download_and_replace_images(markdown_content, html_content, base_url, output_dir="images"):
    """
    Download images found in both Markdown content and original HTML, then replace URLs with local paths
    
    Args:
        markdown_content: Markdown content with image links
        html_content: Original HTML content
        base_url: Base URL for resolving relative image paths
        output_dir: Directory to save downloaded images
        
    Returns:
        Updated markdown content with local image paths
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Find all image URLs in the HTML first (more reliable)
    soup = BeautifulSoup(html_content, 'html.parser')
    img_tags = soup.find_all('img')
    
    # Track downloaded images to avoid duplicates
    downloaded_images = {}
    
    # Process HTML images
    for img in img_tags:
        try:
            img_url = img.get('src', '')
            if not img_url:
                img_url = img.get('data-src', '')
            
            if not img_url or img_url.startswith('data:'):
                continue
                
            # Handle relative URLs
            if not img_url.startswith(('http://', 'https://')):
                img_url = urljoin(base_url, img_url)
            
            # Skip if already downloaded
            if img_url in downloaded_images:
                continue
                
            # Get file extension from URL or default to .jpg
            file_ext = os.path.splitext(urlparse(img_url).path)[1]
            if not file_ext or len(file_ext) > 5:  # Handle invalid extensions
                file_ext = '.jpg'
            
            # Generate a unique filename
            filename = f"{uuid.uuid4().hex}{file_ext}"
            filepath = os.path.join(output_dir, filename)
            
            # Download the image
            print(f"Downloading image: {img_url}")
            response = requests.get(img_url, stream=True, timeout=15)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            # Store the mapping
            downloaded_images[img_url] = os.path.join(output_dir, filename)
            print(f"Saved image to {filepath}")
            
            # Add a small delay to avoid hitting rate limits
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error downloading image {img_url}: {e}")
    
    # Now find and replace image URLs in the Markdown
    image_pattern = r'!\[(.*?)\]\((.*?)\)'
    
    def replace_match(match):
        alt_text = match.group(1)
        img_url = match.group(2)
        
        # If this URL was downloaded, replace with local path
        if img_url in downloaded_images:
            return f'![{alt_text}]({downloaded_images[img_url]})'
        
        # If not found in our downloads, try to download it now
        if not img_url.startswith('data:'):
            try:
                if not img_url.startswith(('http://', 'https://')):
                    img_url = urljoin(base_url, img_url)
                
                file_ext = os.path.splitext(urlparse(img_url).path)[1]
                if not file_ext or len(file_ext) > 5:
                    file_ext = '.jpg'
                
                filename = f"{uuid.uuid4().hex}{file_ext}"
                filepath = os.path.join(output_dir, filename)
                
                response = requests.get(img_url, stream=True, timeout=15)
                response.raise_for_status()
                
                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                print(f"Saved additional image to {filepath}")
                return f'![{alt_text}]({os.path.join(output_dir, filename)})'
            except Exception as e:
                print(f"Error downloading additional image {img_url}: {e}")
        
        # If all else fails, keep the original URL
        return match.group(0)
    
    # Replace all image references
    updated_markdown = re.sub(image_pattern, replace_match, markdown_content)
    
    return updated_markdown


In [17]:
def extract_complete_content(data):
    """
    Make sure we extract the complete content from the data
    """
    # If data is a dictionary, get the content field
    if isinstance(data, dict):
        content = data.get("content", "")
        
        # If content is empty but we have other fields that might contain content
        if not content:
            # Look for other possible content fields
            for field in ["html", "body", "text", "article"]:
                if field in data and data[field]:
                    content = data[field]
                    break
        
        return content
    
    # If data is already a string, return it
    return str(data)


In [19]:
def convert_to_markdown(data, output_file="article.md", download_images=True, images_dir="images"):
    """
    Convert extracted data to Markdown
    
    Args:
        data: Data extracted from magic-html
        output_file: Path to save the markdown file
        download_images: Whether to download images
        images_dir: Directory to save downloaded images
        
    Returns:
        Tuple of (markdown_content, downloaded_image_paths)
    """
    # Configure markdownify options
    markdown_options = {
        'heading_style': 'ATX',  # Use # style headings
        'bullets': '*',  # Use * for unordered lists
        'strong_em_symbol': '**',  # Use ** for bold
        'em_symbol': '*',  # Use * for italics
        'code_language': '',  # Default language for code blocks
        'escape_asterisks': False,  # Don't escape asterisks
        'escape_underscores': False,  # Don't escape underscores
        'strip': None  # Don't strip any elements
    }
    
    # Extract content from the data
    if isinstance(data, dict):
        html_content = extract_complete_content(data)
        title = data.get("title", "Untitled")
        author = data.get("author", "")
        publish_time = data.get("publish_time", "")
        base_url = data.get("base_url", "https://mp.weixin.qq.com")
    else:
        html_content = str(data)
        title = "Untitled"
        author = ""
        publish_time = ""
        base_url = "https://mp.weixin.qq.com"
    
    # Convert HTML to Markdown
    markdown_content = md(html_content, **markdown_options)
    
    # Handle images if needed
    if download_images:
        markdown_content = download_and_replace_images(markdown_content, html_content, base_url, images_dir)
    
    # Create the final markdown content with metadata
    final_markdown = f"# {title}\n\n"
    
    if author:
        final_markdown += f"**Author:** {author}\n\n"
    
    if publish_time:
        final_markdown += f"**Published:** {publish_time}\n\n"
    
    final_markdown += markdown_content
    
    # Write to a Markdown file
    with open(output_file, "w", encoding="utf-8") as file:
        file.write(final_markdown)
    print(f"Markdown content saved to {output_file}")
    
    # Also save the original data as JSON for reference
    with open(f"{os.path.splitext(output_file)[0]}.json", "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
    print(f"Original data saved to {os.path.splitext(output_file)[0]}.json")
    
    # Save the HTML content separately for debugging
    with open(f"{os.path.splitext(output_file)[0]}.html", "w", encoding="utf-8") as file:
        file.write(html_content)
    print(f"HTML content saved to {os.path.splitext(output_file)[0]}.html")
    
    return final_markdown


In [ ]:
# Main execution code - Run this cell to fetch and process the WeChat article

# Initialize the extractor
extractor = GeneralExtractor()

# URL to fetch the WeChat article
url = "https://mp.weixin.qq.com/s?__biz=MzUxODA2MjM0Ng%3D%3D&mid=2247625011&idx=1&sn=042694d7e7aeab0919917dd480358125&chksm=f8e0a3c32938d16d92d4ed94187f6435f2269e902a82321532e38be50fcaa126ab1833c8becc&scene=27"

try:
    # Fetch the HTML content of the page
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
    resp.encoding = resp.apparent_encoding  # Handle encoding
    html = resp.text
    print("HTML fetched successfully.")
    
    # Save original HTML for debugging
    with open("original.html", "w", encoding="utf-8") as f:
        f.write(html)
    print("Original HTML saved to original.html")
    
except requests.RequestException as e:
    print(f"Error fetching the URL: {e}")
    html = ""

# Extract data based on WeChat article type
if html:
    try:
        data = extractor.extract(html, base_url="https://mp.weixin.qq.com", html_type="weixin")
        
        # Add the original HTML to the data for reference
        if isinstance(data, dict):
            data["original_html"] = html
            data["base_url"] = "https://mp.weixin.qq.com"
        
        print("Extraction successful.")
        
        # Get the article title for the filename
        title = data.get("title", "wechat_article")
        safe_title = "".join(c for c in title if c.isalnum() or c in [' ', '-', '_']).strip().replace(' ', '_')
        if len(safe_title) > 50:  # Limit filename length
            safe_title = safe_title[:50]
        
        # Convert to Markdown and save
        output_file = f"{safe_title}.md"
        markdown_content = convert_to_markdown(data, output_file=output_file, download_images=True)
        
        # Display the markdown in the notebook
        print("\nMarkdown Preview:")
        print("-----------------")
        display(Markdown(markdown_content[:2000] + "..." if len(markdown_content) > 2000 else markdown_content))
            
    except Exception as e:
        print(f"Error during extraction or conversion: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No HTML content to process.")


In [ ]:
# Optional: Display the saved markdown file
# Run this cell after the previous one completes to display the full markdown

try:
    with open(f"{safe_title}.md", 'r', encoding='utf-8') as f:
        markdown_content = f.read()
    
    display(Markdown(markdown_content))
except NameError:
    print("Please run the previous cell first to generate the markdown file.")
except FileNotFoundError:
    print("Markdown file not found. Please run the previous cell first.")


In [ ]:
# Optional: Display the images that were downloaded
# Run this cell to see what images were saved

import matplotlib.pyplot as plt
from PIL import Image
import glob

try:
    image_files = glob.glob('images/*.jpg') + glob.glob('images/*.png') + glob.glob('images/*.gif') + glob.glob('images/*.jpeg')
    
    if not image_files:
        print("No images found in the images directory.")
    else:
        print(f"Found {len(image_files)} images:")
        
        # Display up to 5 images
        for i, img_path in enumerate(image_files[:5]):
            print(f"Image {i+1}: {img_path}")
            img = Image.open(img_path)
            plt.figure(figsize=(8, 8))
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"Image {i+1}: {os.path.basename(img_path)}")
            plt.show()
        
        if len(image_files) > 5:
            print(f"...and {len(image_files) - 5} more images")
except Exception as e:
    print(f"Error displaying images: {e}")
